In [1]:
import numpy as np
import pandas as pd

In [8]:
# draw times from two distinct exponential distributions
# and sum them (one 'cycle' consists of 2 consecutive exponential processes)
rate1 = 1
rate2 = 2

cycle_times = []
for i in range(1000000):
    time1 = np.random.exponential(rate1, size=1)
    time2 = np.random.exponential(rate2, size=2)
    cycle_time = time1[0] + time2[0]
    cycle_times.append(cycle_time)
    
# the time points of cycle completions in continuous time trace 
time_points = np.cumsum(cycle_times)

# convert to pandas.timedelta
time_deltas = pd.to_timedelta(time_points, unit='s')

In [9]:
# count events every 20 seconds
resample = '20 s'
events = np.ones(len(time_deltas))
series = pd.Series(events, index=time_deltas)
series = series.resample(resample).sum()
time_deltas = series.index
in_seconds = time_deltas / np.timedelta64(1, "s")
series.index = in_seconds

In [10]:
series

3.657842e+00    4.0
2.365784e+01    9.0
4.365784e+01    5.0
6.365784e+01    7.0
8.365784e+01    6.0
               ... 
3.002944e+06    7.0
3.002964e+06    6.0
3.002984e+06    5.0
3.003004e+06    6.0
3.003024e+06    5.0
Length: 150152, dtype: float64

In [11]:
# probability of counting e.g., 5 events in 20 seconds
fives = np.where(series == 5)[0]
pr_five = len(fives)/len(series)
print(f'The observed probability of counting 5 cycles within 20 seconds (i.e., events) is {pr_five:.3f}')

The observed probability of counting 5 cycles within 20 seconds (i.e., events) is 0.154


In [12]:
# defining the proposed mathematical expression
from mpmath import nsum, inf, factorial
from scipy.special import gamma

def phi(b, c, w, z):
    f = b**2/c
    result = float(nsum(lambda k, l: f * w**k * z**l / (factorial(l)*factorial(k)), [0, inf], [0, inf]))

    return result


def c_x(t, a, b, x):
    part1 = (a * b)**x
    part2 = t**(2*x)/gamma(1+2*x)
    part3 = phi(x, 1+2*x, -t*a, -t*b)
    func = part1 * part2 * part3

    return func


def counts_hypo_interarrival_times_pmf(t, a, b, x):
    pmf = c_x(t, a, b, x) - c_x(t, a, b, x+1)

    return pmf

In [14]:
# inserting the values into the proposed mathematical expression
t = 20
a = 1
b = 2
x = 5
counts_hypo_interarrival_times_pmf(t, a, b, x)

-1.1473533046143436e-17